# CUB — Data Analysis  (full CUB **and** CUB70)

Set `DATASET` below. **Sections 1–4 apply to both.** Section 5 is **CUB70-only**.

**Why CUB70 exists (not just fewer classes):** CUB70 = the first 70 CUB classes that ship **per-part segmentation masks**. Masks let you measure part visibility / occlusion and do **part-level grounding on real birds** — the CUB analogue of the FunnyBirds deletion test. **Full CUB (200) has no part masks**, so it only supports the recall-gap axis (attributes varying within a species), not occlusion/grounding.

In [1]:
import os, sys, pickle
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent   # run from curated/notebooks
# ---- CONFIG ----
DATASET = "cub70"   # "cub" = full 200 (recall-gap axis) | "cub70" = 70 classes WITH part masks (occlusion/grounding)
DIRS = {"cub":   CURATED/"CUB_processed"/"class_attr_data_10",
        "cub70": CURATED/"CUB_processed"/"class_attr_data_10_cub70_original"}
PKLS_DIR = DIRS[DATASET]; ATTR_DIR = CURATED/"CUB_200_2011"
# ----------------
def load(f):
    p = PKLS_DIR/f
    if not p.exists():
        avail=[d.name for d in (CURATED/"CUB_processed").glob("*")] if (CURATED/"CUB_processed").exists() else []
        raise FileNotFoundError(f"{p} missing (DATASET={DATASET!r}). Available under CUB_processed: {avail}")
    return pickle.load(open(p,"rb"))
tr=load("train.pkl"); te=load("test.pkl")
Atr=np.array([r["attribute_label"] for r in tr]); ytr=np.array([r["class_label"] for r in tr])
Ate=np.array([r["attribute_label"] for r in te]); yte=np.array([r["class_label"] for r in te])
nC=Atr.shape[1]
print(f"{DATASET}: train {len(tr)} / test {len(te)} imgs · {len(set(ytr))} species · {nC} attributes")

names=groups=None
try:
    sys.path.insert(0, str(REPO/"external"/"minimal_cbm"))
    from src.datasets.cub200 import USED_ATTRIBUTES
    alln=[l.split(" ",1)[1].strip() for l in open(ATTR_DIR/"attributes.txt") if l.strip()]
    names=[alln[i-1] for i in USED_ATTRIBUTES]
    groups={}; [groups.setdefault(n.split("::")[0],[]).append(j) for j,n in enumerate(names)]
    print(f"{len(names)} attrs in {len(groups)} groups")
except Exception as e:
    print("names/groups unavailable (indices only):", e)

FileNotFoundError: /scratch/network/cr7998/cv_emergence_project/curated_data/CUB_processed/class_attr_data_10_cub70_original/train.pkl missing (DATASET='cub70'). Available under CUB_processed: ['class_attr_data_10', 'class_attr_data_10_relabeled', 'class_attr_data_10_cub70_original']

## 1. Class balance

In [ ]:
tc=pd.Series(ytr).value_counts().sort_index(); ec=pd.Series(yte).value_counts().sort_index()
print("train/species:",tc.min(),"-",tc.max(),"| test/species:",ec.min(),"-",ec.max())
fig,ax=plt.subplots(1,2,figsize=(10,2.6))
ax[0].bar(tc.index,tc.values); ax[0].set_title("train / species")
ax[1].bar(ec.index,ec.values,color="tab:orange"); ax[1].set_title("test / species"); plt.tight_layout()

## 2. Attribute prevalence

In [ ]:
prev=Atr.mean(0)
fig,ax=plt.subplots(figsize=(12,3)); ax.bar(range(nC),prev); ax.set_ylabel("P(=1)"); ax.set_xlabel("attribute")
print("prevalence: min %.3f  max %.3f  mean %.3f"%(prev.min(),prev.max(),prev.mean()))

## 3. Class × attribute matrix

In [ ]:
M=pd.DataFrame(Atr).assign(c=ytr).groupby("c").mean().values
print("cells exactly 0/1:",round(float(np.mean((M==0)|(M==1))),4),"(1.0 = class-level/constant labels)")
fig,ax=plt.subplots(figsize=(11,5)); im=ax.imshow(M,aspect="auto",cmap="magma",vmin=0,vmax=1)
ax.set_xlabel("attribute"); ax.set_ylabel("species"); fig.colorbar(im,ax=ax,fraction=0.02)

## 4. Species-constancy — is the recall gap powered here?
Within-species std of each attribute on test. **>0 ⇒ attributes vary within a species ⇒ matched-pair recall gap has real signal** (unlike FunnyBirds ~0). ~0 ⇒ these pkls are class-level labels ⇒ recall gap underpowered → use deletion/grounding.

In [ ]:
within=np.array([Ate[yte==c].std(0) for c in np.unique(yte)])
frac0=float(np.mean(within==0)); nimg=int(np.median([(yte==c).sum() for c in np.unique(yte)]))
print(f"(species,attr) within-species std==0: {frac0:.4f} | mean within-species std {within.mean():.4g} | median test imgs/species {nimg}")
print("VERDICT:", "within-species variation EXISTS -> recall gap testable (image-level labels)" if frac0<0.99
      else "species-constant (class-level labels) -> recall gap underpowered; use deletion/grounding")

## 5. CUB70 ONLY — part segmentation / visibility
**This is why we use CUB70.** Per-part mask area from the segmentation masks
(`cub70_visibility.parquet`): candidate occlusion features per body part, and the
basis for part-level grounding on real birds. Full CUB has no masks → this section
is empty for `DATASET="cub"`.

In [ ]:
visp = CURATED/"cub70_visibility.parquet"
if DATASET=="cub70" and visp.exists():
    vis=pd.read_parquet(visp)
    prof=(vis.groupby("part").agg(mean_area_frac=("area_frac","mean"),
          frac_visible=("visible","mean"), n=("visible","size"))
          .reset_index().sort_values("frac_visible"))
    display(prof)
    fig,ax=plt.subplots(figsize=(6,3)); ax.bar(prof.part,prof.frac_visible)
    ax.set_ylabel("frac visible"); ax.set_title("CUB70 per-part visibility (low = occlusion-prone)")
    plt.xticks(rotation=45,ha="right")
    print("Low frac_visible / small area = candidate occlusion-prone parts to line up vs measured backwash.")
elif DATASET=="cub70":
    print("cub70_visibility.parquet not found -> build it: python data/cub70/build_cub70_visibility.py")
else:
    print("Segmentation is CUB70-only. Full CUB has no part masks -> recall-gap axis only, no occlusion/grounding.")